 Naver open API를 활용하여 네이버지식인 "전주여행"과 "경주여행"을 검색
 -> 품사태깅 백업
 -> 명사만 추출(re?) -> 빈도분석(DataFrame) -> 빈도 시각화(워드클라우드, Text)
 -> Word2Vec

# 1. 네이버 open API를 활용하여 검색 추출
- 검색어, no, title, link, description, title + ' ' + description(total_text)

In [ ]:
'''
import urllib.request
import requests
import sys
import pandas as pd
from dotenv import load_dotenv
import os
from bs4 import BeautifulSoup
    
load_dotenv()

def get_naver_kin(keyword, cnt):
    
     
    client_id = os.getenv('Client_ID')
    client_secret = os.getenv('Client_Secret')

    url = f'https://openapi.naver.com/v1/search/kin.json?query={keyword}&display={cnt}'
    headers = {'X-Naver-Client-Id': client_id,
               'X-Naver-Client-Secret':client_secret}
    response = requests.get(url, headers=headers)

    items = response.json()['items']
    items_list = []
    for item in items:
        items_list.append([
            item.get('title').replace('<b>','').replace('</b>',''),
            item.get('link'),
            item.get('description').replace('<b>','').replace('</b>','')
        ])
    return pd.DataFrame(items_list, columns=['title','link','description'])

df_jeonju = get_naver_kin("전주여행", 100)
df_gyeongju = get_naver_kin("경주여행", 100)

df_jeonju['text'] = df_jeonju['title'] + ' ' + df_jeonju['description']
df_gyeongju['text'] = df_gyeongju['title'] + ' ' + df_gyeongju['description']

print("전주여행 검색 :", df_jeonju['text'].iloc[0])
print("경주여행 검색 :", df_gyeongju['text'].iloc[0])
'''


In [ ]:
import urllib.request
import requests
import sys
import pandas as pd
from dotenv import load_dotenv
import os
from bs4 import BeautifulSoup
import time

load_dotenv()

def get_naver_kin(keyword, total_cnt):
    client_id = os.getenv('Client_ID')
    client_secret = os.getenv('Client_Secret')

    items_list = []
    for start in range(1, total_cnt + 1, 100):
        display = min(100, total_cnt - start + 1)
        url = f'https://openapi.naver.com/v1/search/kin.json?query={keyword}&display={display}&start={start}'
        headers = {
            'X-Naver-Client-Id': client_id,
            'X-Naver-Client-Secret': client_secret
        }

        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"❌ Error {response.status_code} at start={start}")
            continue

        items = response.json().get('items', [])
        for item in items:
            items_list.append([
                item.get('title', '').replace('<b>', '').replace('</b>', ''),
                item.get('link', ''),
                item.get('description', '').replace('<b>', '').replace('</b>', '')
            ])
        
        time.sleep(0.5)

    return pd.DataFrame(items_list, columns=['title', 'link', 'description'])

df_jeonju = get_naver_kin("전주여행", 500)
df_gyeongju = get_naver_kin("경주여행", 500)

df_jeonju['text'] = df_jeonju['title'] + ' ' + df_jeonju['description']
df_gyeongju['text'] = df_gyeongju['title'] + ' ' + df_gyeongju['description']

print("전주여행 총 수:", len(df_jeonju))
print("경주여행 총 수:", len(df_gyeongju))
print("전주여행 검색 :", df_jeonju['text'].iloc[0])
print("경주여행 검색 :", df_gyeongju['text'].iloc[0])


In [ ]:
get_naver_kin("경주여행",5)

In [ ]:
# 방법1 (자동 명사 추출)
from konlpy.tag import Kkma
kkma = Kkma(jvmpath=None,
            max_heap_size=1024)

nouns_jeonju = [kkma.nouns(text) for text in df_jeonju['text']]
nouns_gyeongju = [kkma.nouns(text) for text in df_gyeongju['text']]

print("전주여행 예시:", nouns_jeonju[0])
print("경주여행 예시:", nouns_gyeongju[0])


In [ ]:
# 방법2 (N으로 시작하는)
from konlpy.tag import Kkma
kkma = Kkma(jvmpath=None,
            max_heap_size=1024)

nouns_jeonju = [
    [word for word, tag in kkma.pos(text) if tag.startswith('N')]
    for text in df_jeonju['text']
]

nouns_gyeongju = [
    [word for word, tag in kkma.pos(text) if tag.startswith('N')]
    for text in df_gyeongju['text']
]

print("전주여행 예시:", nouns_jeonju[0])
print("경주여행 예시:", nouns_gyeongju[0])

In [ ]:
nouns_jeonju, nouns_gyeongju

In [ ]:
import pandas as pd

jeonju_list = [noun for sublist in nouns_jeonju for noun in sublist if len(noun) > 1]
gyeongju_list = [noun for sublist in nouns_gyeongju for noun in sublist if len(noun) > 1]

df_jeonju_nouns = pd.DataFrame({'word': jeonju_list})
df_freq_jeonju = df_jeonju_nouns['word'].value_counts().reset_index()
df_freq_jeonju.columns = ['word', 'freq']

df_gyeongju_nouns = pd.DataFrame({'word': gyeongju_list})
df_freq_gyeongju = df_gyeongju_nouns['word'].value_counts().reset_index()
df_freq_gyeongju.columns = ['word', 'freq']

df_freq_jeonju.head(10)
df_freq_gyeongju.head(10)

In [ ]:
# from wordcloud import STOPWORDS
# 불용어 = STOPWORDS | {'추천', '법률'} # | : 집합합연산자
# 불용어 = set(['대통령','법률'])
불용어 = {'추천','질문','계획','여행','전주','경주','일정','숙소'}
불용어

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wordcloud1 = WordCloud(font_path='data/NanumPenScript-Regular.ttf',
                             background_color='white',
                             max_words=2000,
                            #  relative_scaling=0.9,
                             stopwords=불용어,
                             width=800,
                             height=400)
wordcloud1 = wordcloud1.generate(' '.join(jeonju_list))

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud1)
plt.axis('off')
plt.show()


In [ ]:
wordcloud2 = WordCloud(font_path='data/NanumPenScript-Regular.ttf',
                             background_color='white',
                             max_words=2000,
                            #  relative_scaling=0.9,
                             stopwords=불용어,
                             width=800,
                             height=400)

wordcloud2 = wordcloud2.generate(' '.join(gyeongju_list))

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud2)
plt.axis('off')
plt.show()


In [ ]:
from gensim.models import Word2Vec

model_jeonju = Word2Vec(sentences=nouns_jeonju,
                        vector_size=100,
                        window=5,
                        min_count=2,
                        workers=1)

model_gyeongju = Word2Vec(sentences=nouns_gyeongju,
                          vector_size=100,
                          window=5,
                          min_count=2,
                          workers=1)


In [ ]:
print("'전주'와 유사한 단어:")
print(model_jeonju.wv.most_similar('전주'))

print("'경주'와 유사한 단어:")
print(model_gyeongju.wv.most_similar('경주'))
